# 代码学习笔记

这份笔记配合仓库里已经补充好的行尾学习注释一起使用，目标是帮助你：

- 按顺序读懂 `classic/` 主训练线
- 快速定位重要调参入口
- 理解 `warp_gpu/` 这条 GPU 训练线的职责
- 在训练、测试和继续训练时知道先看哪里


## 推荐阅读顺序

### 1. 先读 `classic/env.py`

重点看：

- `MujocoEnvConfig`
- `reset`
- `step`
- 奖励参数和课程学习参数

这一步主要回答两个问题：

1. 状态是怎么构造的？
2. 奖励为什么这样设计？

### 2. 再读 `classic/train.py`

重点看：

- `TrainArgs`
- `_make_env`
- `_build_model`
- `train`
- `test`

这一步主要回答两个问题：

1. 参数从命令行是怎么流进环境和模型的？
2. 训练产物保存在哪里？

### 3. 然后读 `classic/test.py`

它最适合用来理解：

- 如何做最小 smoke test
- `VecNormalize` 在推理时怎么加载
- 测试时哪些参数必须与训练保持一致

### 4. 最后再看 `warp_gpu/`

`warp_gpu/` 不是主实验线，建议把它当作：

- Playground 接入层
- Warp 运行时检查层
- Warp GPU 训练入口


## 最关键的调参入口

### 环境参数

主要在 `classic/env.py -> MujocoEnvConfig`：

- `success_threshold`
- `max_steps`
- `torque_low / torque_high`
- `curriculum_stage1_fixed_episodes`
- `curriculum_stage2_random_episodes`
- `curriculum_stage2_range_scale`
- 奖励相关的 `*_penalty` / `*_reward` / `*_gain`

### 训练参数

主要在 `classic/train.py -> TrainArgs`：

- `algo`
- `timesteps`
- `n_envs`
- `batch_size`
- `buffer_size`
- `gradient_steps`
- `learning_starts`
- `eval_freq`
- `physics_backend`
- `legacy_zero_ee_velocity`

### 调参建议

- 先固定 `physics_backend=mujoco`
- 先用 `--eval-freq 0` 做快速迭代
- 先把课程学习跑通，再动奖励项
- 每次只改一组参数，不要同时改奖励和网络结构


## 这次代码注释怎么用

仓库里的 Python 源码现在已经补充了大量行尾学习注释，建议这样用：

1. 先快速扫文件结构和函数名
2. 再逐段读行尾注释，先理解每段职责
3. 最后回到参数定义位置，建立“参数 -> 行为”的映射

如果你在读的时候觉得注释太密，可以优先看这些文件：

- `classic/env.py`
- `classic/train.py`
- `classic/test.py`


## 常用命令

随机策略测试：

```bash
python -m classic.test --random-policy --episodes 1 --max-steps 10 --no-render
```

训练主线：

```bash
python classic/train.py --algo sac --robot ur5_cxy --run-name exp_sac --timesteps 500000 --device cuda --no-render
```

测试模型：

```bash
python classic/test.py --algo sac --robot ur5_cxy --run-name exp_sac --episodes 3 --render
```

复现 zeroarm 旧速度语义：

```bash
python classic/train.py --algo sac --robot zero_robotiq --legacy-zero-ee-velocity --no-render
```
